In [ ]:
import pandas as pd


In [ ]:
from src.Pair_Selection import *
from src.Cointegration import find_cointegrated_pairs

prices = load_prices("data/processed/prices.parquet")

split = int(len(prices) * 0.7)

train_prices = prices.iloc[:split]
test_prices = prices.iloc[split:]

# Candidate screening must use formation data only.
# Using test_prices here would leak out-of-sample information into pair selection.
returns = compute_returns(train_prices)
corr = correlation_matrix(returns)

candidate_pairs = generate_candidate_pairs(
    corr,
    top_n=10,
)

candidate_summary(candidate_pairs)
candidate_pairs[:20]

cointegrated_pairs, spreads = find_cointegrated_pairs(
    train_prices,
    candidate_pairs,
    significance=0.01,
)

print(cointegrated_pairs.head())
print(f"\nCointegrated pairs found: {len(cointegrated_pairs)}")


In [ ]:
import matplotlib.pyplot as plt

print(cointegrated_pairs["pvalue"].describe())
plt.hist(cointegrated_pairs["pvalue"], bins=30)
plt.show()


In [ ]:
train_prices.to_parquet("data/processed/train_prices.parquet", index=True)
test_prices.to_parquet("data/processed/test_prices.parquet", index=True)
cointegrated_pairs.to_parquet("data/processed/cointegrated_pairs.parquet", index=True)


In [ ]:
import pickle

with open("data/processed/spreads.pkl", "wb") as f:
    pickle.dump(spreads, f)
